# PySpark Medallion Architecture & ML Pipeline
This notebook implements a Bronze -> Silver -> Gold data architecture and an ML Pipeline for Sentiment Analysis using the IMDB Dataset.

### 1. Initialization
Set up the Spark session and import necessary libraries.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Initialize Spark Session
spark = SparkSession.builder \
    .appName('IMDB_Medallion_ML') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark

### 2. Bronze Layer
Read the raw `IMDB Dataset.csv` and save it to the `data/bronze/` directory.

In [ ]:
# Read the raw CSV data
df_raw = spark.read.csv('IMDB Dataset.csv', header=True, inferSchema=True)

# Show raw data
print('Raw Data Schema:')
df_raw.printSchema()
df_raw.show(5)

# Save to Bronze layer
bronze_path = 'data/bronze'
df_raw.write.mode('overwrite').parquet(bronze_path)
print(f'Bronze layer saved to {bronze_path}')

### 3. Silver Layer
Read from Bronze, clean the text (remove HTML, lowercase, regex), map sentiment to numeric, and save to `data/silver/`.

In [ ]:
# Read from Bronze layer
df_bronze = spark.read.parquet(bronze_path)

# Clean text and map sentiment
# 1. Remove HTML tags like <br />
# 2. Remove punctuation and special characters
# 3. Lowercase
df_cleaned = df_bronze.withColumn('clean_review', F.regexp_replace('review', '<[^>]+>', ' ')) \
                      .withColumn('clean_review', F.regexp_replace('clean_review', '[^a-zA-Z\\s]', '')) \
                      .withColumn('clean_review', F.lower(F.col('clean_review'))) \
                      .withColumn('label', F.when(F.col('sentiment') == 'positive', 1.0).otherwise(0.0))

# Filter out any nulls
df_silver = df_cleaned.filter(F.col('clean_review').isNotNull()).select('clean_review', 'label')

df_silver.show(5)

# Save to Silver layer
silver_path = 'data/silver'
df_silver.write.mode('overwrite').parquet(silver_path)
print(f'Silver layer saved to {silver_path}')

### 4. Gold Layer
The Silver layer is already feature-ready for ML, but we will formalize the Gold layer by saving the finalized dataset here. This layer represents the highly refined data.

In [ ]:
# Read from Silver layer
df_silver_read = spark.read.parquet(silver_path)

# Save to Gold layer
gold_path = 'data/gold'
df_silver_read.write.mode('overwrite').parquet(gold_path)
print(f'Gold layer saved to {gold_path}')

# Load Gold Layer for ML
df_gold = spark.read.parquet(gold_path)

### 5. Machine Learning Pipeline
Define the Tokenizer, StopWordsRemover, HashingTF, IDF, and LogisticRegression in a PySpark ML Pipeline. Train on 80% and evaluate on 20%.

In [ ]:
# Split data
train_data, test_data = df_gold.randomSplit([0.8, 0.2], seed=42)

# Define Pipeline Stages
tokenizer = Tokenizer(inputCol='clean_review', outputCol='words')
remover = StopWordsRemover(inputCol='words', outputCol='filtered_words')
hashingTF = HashingTF(inputCol='filtered_words', outputCol='rawFeatures', numFeatures=10000)
idf = IDF(inputCol='rawFeatures', outputCol='features')
lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=10)

pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, lr])

# Train Model
print('Training model...')
model = pipeline.fit(train_data)
print('Model training complete.')

# Make Predictions
predictions = model.transform(test_data)
predictions.select('clean_review', 'label', 'prediction', 'probability').show(5)

### 6. Evaluation
Evaluate the model's accuracy and Area Under ROC.

In [ ]:
# Evaluate accuracy
evaluator_acc = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
accuracy = evaluator_acc.evaluate(predictions)
print(f'Accuracy: {accuracy:.4f}')

# Evaluate AUC
evaluator_auc = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')
auc = evaluator_auc.evaluate(predictions)
print(f'Area Under ROC: {auc:.4f}')